# Simple Simulation (single plane)

This notebook serves to get familiar with the simulator. The UAV here is a **Zephyr** fixed-wing (ArduPlane).

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import DATA_PATH, Color, Model
from simulator.entities import SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.planner import AutoPlan, GuidedPlan, Plan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
)

clean()


## Simulation Positions

1. **Set the simulation origin in GRA (Global Geodetic Reference) and ENU coordinates**  
   * The **GRA (Global Geodetic Reference)** origin ties the simulation to a specific geographic location on Earth, defined by latitude, longitude, and altitude.
   In contrast, the ENU (East–North–Up) coordinate frame is a local reference system that is not tied to any particular Earth position.
   While ArduPilot internally uses global geodetic (GRA) coordinates, ENU coordinates are earth-agnostic and easier to interpret and manipulate mathematically.
   They are also consistent with Gazebo’s coordinate convention. Typically, the ENU origin is defined  $(x = 0, y = 0)$.
   * 	The heading attribute in both GRAPose and ENUPose defines the orientation of the UAV when specifying drone positions.
   Additionally, it is used to rotate local ENU coordinates whenever positions are defined relative to a posed coordinate, ensuring spatial consistency between global and local frames.
2. **Set UAV base home and path**  
   * **Base home**: The starting UAV position (relative to the simulation origin). It is given in ENU coordinates for easier use.  
   * **Base path**: The UAV trajectory (relative to the uav base home). It is also given in ENU coordinates for easier use.

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)


In [ ]:
base_home = ENUPose(x=5, y=10, z=0, heading=90)
side_len = 100
base_path = Plan.create_square_path(side_len, alt=20, clockwise=False)


## Convert relative to absolute positions
This can be done implicitly using the helper methods:
```python
SimVehicle.from_relative()
AutoPlan.square_traj()
```

In [ ]:
enu_home = enu_origin.to_abs(base_home)
gra_home = gra_origin.to_abs(base_home)
enu_waypoints = enu_home.to_abs_all(base_path)
gra_waypoints = gra_home.to_abs_all(base_path)


## Create Vehicle

1. **Set identifiers and display**  
   * **`sysid`**: Unique MAVLink system ID for the UAV (must be unique per vehicle).  
   * **`color`**: Display color used for the UAV and its waypoints/paths in Gazebo or plots.

In [ ]:
sysid = 1
color = Color.ORANGE
speed = 14
model = Model.ZEPHYR



2. **Build the plan**   
   * **`AutoPlan(name, mission_path)`** is the **default plan type used in this project** for executing **ArduPilot AUTO missions**..  
      - **`mission_path`**: Path to a saved mission file (e.g., `mission_1.waypoints`). This file is used to upload **prebuilt missions**, and reused by the `save_*` helper methods when generating missions programmatically.
      - `AutoPlan` extends `Plan` and **automatically performs** the following steps:
         - uploads the mission file (`make_upload_mission(mission_path)`),
         - arms and configures the vehicle (`Plan.arm()`),
         - starts the mission (`make_start_mission()`), and
         - enables mission monitoring (`make_monitoring()`).

      - Use **`AutoPlan.save_basic_mission(sysid, gra_wps)`** to **generate and save** a minimal mission file containing:
         - HOME,
         - TAKEOFF,
         - navigation waypoints, and
         - LAND,  
      from **absolute GRA waypoints**.
   * You can also create **`GuidedPlan`** instances, where the onboard logic **sends MAVLink commands sequentially**, waiting for each command to complete before issuing the next one.

   * Finally, you can **subclass `Plan`** to define **custom mission behaviors**, combining actions and steps to implement more advanced or specialized flight logic.

### Auto Plan 

In [ ]:
mission_dir = DATA_PATH / "missions"
mission_dir.mkdir(parents=True, exist_ok=True)
mission_path = str(mission_dir / f"mission_{sysid}.waypoints")
AutoPlan.save_basic_mission(
    mission_path=mission_path,
    sysid=sysid,
    gra_wps=GRAPose.unpose_all(gra_waypoints),
    land=True,
)
auto_plan = AutoPlan(
    name="simple_auto_plan",
    mission_path=mission_path,
    firmware=model.firmware,
    navigation_speed=speed,
)


In [ ]:
auto_plan


### Guided Plan 

In [ ]:
guided_plan = GuidedPlan(
    name="simple_guided_plan",
    wps=ENUPose.unpose_all(enu_waypoints),
    firmware=model.firmware,
    autoland_alt=10,
    wp_margin=30,
    autoland_wp_dist=side_len - 40,
)
guided_plan



3. **Instantiate the vehicle**  
   * **`SimVehicle(`**  
     `model=model` - Vehicle model and firmware configuration.  
     `sysid=sysid` - MAVLink system ID.  
     `plan=guided_plan` - Mission plan to execute; use `auto_plan` to run the saved AUTO mission instead.  
     `color=color` - Visualization color.  
     `home=enu_home` - Home position in **ENU**; its **heading** defines the UAV's initial yaw.  
     `waypoints` - Waypoints the UAV must follow. Although conceptually tied to the plan, this field is used **only for visualization** and should not be confused with the UAV's actual mission logic. In an **`AutoPlan`**, the mission may also include **non-waypoint actions** such as arming, speed changes, or monitoring.  
     `parm=...` — *(optional)* SITL parameter file for this vehicle, appended to the firmware defaults. Defaults to `params/vehicle.parm`.  

4. **`pose` / `unpose` semantics**  
   * **`pose`**: Adds a heading to GRA or ENU coordinates, returning a `GRAPose` or `ENUPose` object, respectively. The default heading is `0`.  
   * **`unpose` / `unpose_all`**: Inverse of `pose`. The `_all` version operates on lists of coordinates or poses.

5. **Multi-UAV note**  
   * Repeat this block for each UAV with a **distinct `sysid`** and typically a different **`home`** to avoid ID or spawn conflicts.

In [ ]:
veh = SimVehicle(
    model=model,
    sysid=sysid,
    plan=guided_plan,  # auto_plan,  #
    color=color,
    home=enu_home,
    waypoints=ENUPose.unpose_all(enu_waypoints),
)


## Visualizer

We can choose between three visualization modes:
* Gazebo
* QGroundControl
* No visualization

Below we describe how to configure each of them for completeness.

### Gazebo

The **Gazebo** visualizer provides a **realistic 3D simulation environment**.  
1. **Set up Gazebo**  
   * **`gra_origin`**: The global reference position (latitude, longitude, altitude, heading) used to tie the simulation to a configurable Gazebo world.  
   * **`world_path`**: The path to the Gazebo world file to load (e.g., `"simulator/visualizer/gazebo/worlds/runway.world"`).  
     The simulator automatically updates this world file by inserting UAV models and visual markers during initialization.

2. **Add optional markers**  
   * Markers can be manually added for visualization or debugging purposes using the `GazMarker` class.  
     Each marker has attributes such as **name**, **group**, **position**, and **color**.  
     They can be used, for instance, to mark reference points like the simulation origin or target locations.  

3. **Automatic waypoint markers**  
    * When the simulator starts, the Gazebo visualizer automatically generates color-coded waypoint markers for each UAV based on the **`waypoints` argument** in its `SimVehicle` definition.  


In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)
gaz.markers.append(origin_gaz)


### QGroundControl

The **QGroundControl (QGC)** visualizer provides a 2D mission and telemetry view of the simulation.
* `gra_origin` anchors the simulated vehicles on QGC's map.
* Add map markers with `QGCMarker` objects appended to `qgc.markers`.

In [ ]:
qgc = QGC(gra_origin)
origin_qgc = QGCMarker(name="origin", pos=gra_origin.unpose(), color=Color.WHITE)
qgc.markers.append(origin_qgc)


### No Visualizer

Run the simulator **headless** (no GUI).
* Uses `gra_origin` to place each UAV's home — SITL is always given `--home`.
* Registers vehicles normally but **does not render** markers or paths, and adds no SITL arguments of its own.
* Ideal for batch runs or large-scale simulations where many UAVs are run without graphical rendering.

In [ ]:
novis = NoVisualizer(gra_origin)

## Oracle — the scenario

The `Oracle` holds *what* is simulated, and nothing about how it runs:

* **Vehicles** — `orac.add_vehicle(veh)`.
* **GCSs** — `orac.add_gcs(gcs)`. A vehicle may be monitored by no GCS, one, or several.
* **Adversarial setup** — `orac.intervention[sysid]`, `orac.mitm[sysid]`, both keyed by sysid.
* **Tuning** — `Oracle(transmission_range=...)` sets the inter-vehicle Remote ID range in metres (default 100).

At run time it is also the coordinator: it collects each vehicle's Remote ID,
tracks which vehicles are near which, and relays updates to those in range.

The scenario is built first and is complete on its own — no simulator involved.

In [ ]:
orac = Oracle()

orac.add_vehicle(veh)

## Simulator — the machinery

The `Simulator` decides *how* the scenario is executed. It owns no scenario
state of its own; it reads everything from the `oracle` it is given.

* **`oracle`** — required: the scenario to run.
* **`visualizer`** — `Gazebo`, `QGC` or `NoVisualizer`.
* **`terminals`** — which processes open a visible terminal (e.g. `[SimProcess.LOGIC]`).
* **`suppress_output`** — which processes have their output discarded (defaults to SITL and the ADS-B socat).
* **`verbose`** — logging level.

`simulator.preview()` renders the configured scenario before anything is launched.

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,  # novis,  #
    terminals=[SimProcess.LOGIC],
    verbose=1,
)

simulator.preview()

## Run

1. **Launch**
   * `simulator.launch()` starts SITL, logic, and the selected visualizer.
   * The `Oracle` was built *before* the simulator and passed to it, so `launch()` binds it to the run once every port offset is known — no wiring needed here.

2. **Execute**
   * `orac.run()` blocks until all UAV missions finish.
   * Console logs show lifecycle events such as visualizer mode, PIDs, mission DONE, and shutdown.

> The two steps are kept apart here so you can watch the visualizer come up before anything flies. When you don't need that, `simulator.run()` does both.


In [ ]:
simulator.launch()

In [ ]:
orac.run()

## Plot the recorded trajectory

The Oracle keeps the vehicle's Remote ID track while the run proceeds, so plotting needs
no extra setup.

* `orac.plot_trajectories()` — the flown track in the local ENU frame, coloured by the
  vehicle's own `SimVehicle.color`.
* `save="run.png"`, `show=False` — write to a file instead of blocking the cell.

Samples taken before the EKF converges (GPS still reading 0,0) are skipped, so the plot
starts where the vehicle actually got a fix. `Oracle(record_positions=False)` turns
recording off.

In [ ]:
orac.plot_trajectories()